In [1]:
!pip install langchain
!pip install sentence-transformers
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 28.8 MB/s eta 0:00:00


In [2]:
documents = [

"""
Contract A:
Employee receives 20 days annual leave.
Notice period is 60 days.
""",

"""
Contract B:
Health insurance is provided.
Notice period is 30 days.
""",

"""
Contract C:
Bonus eligibility starts after 1 year.
"""

]

print(len(documents))

3


In [3]:
chunks = []

for doc in documents:

    words = doc.split()

    for i in range(0,len(words),50):

        chunk = " ".join(
            words[i:i+50]
        )

        chunks.append(chunk)

print(chunks)

['Contract A: Employee receives 20 days annual leave. Notice period is 60 days.', 'Contract B: Health insurance is provided. Notice period is 30 days.', 'Contract C: Bonus eligibility starts after 1 year.']


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = model.encode(
    chunks
)

print(embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(3, 384)


In [5]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(embeddings)
)

print(
    "Vectors stored:",
    index.ntotal
)

Vectors stored: 3


In [6]:
def retrieve(query):

    query_embedding = model.encode(
        [query]
    )

    D,I = index.search(
        np.array(query_embedding),
        k=2
    )

    results = []

    for idx in I[0]:

        results.append(
            chunks[idx]
        )

    return results

In [7]:
query = "What is the notice period?"

results = retrieve(query)

for r in results:
    print(r)

Contract B: Health insurance is provided. Notice period is 30 days.
Contract A: Employee receives 20 days annual leave. Notice period is 60 days.


In [8]:
context = "\n".join(
    retrieve(
        "What is the notice period?"
    )
)

PROMPT = f"""
You are a legal assistant.

Answer only using context.

CONTEXT:
{context}

QUESTION:
What is the notice period?
"""

print(PROMPT)


You are a legal assistant.

Answer only using context.

CONTEXT:
Contract B: Health insurance is provided. Notice period is 30 days.
Contract A: Employee receives 20 days annual leave. Notice period is 60 days.

QUESTION:
What is the notice period?



In [9]:
answer = retrieve(
    "What is the notice period?"
)

print("Answer:")
print(answer[0])

Answer:
Contract B: Health insurance is provided. Notice period is 30 days.
